# QC Style Consistency Run

Generate the three artifacts used to inspect consistency across the Fulcher overview, line-fit QC page, and Boltzmann QC plot for one frame.

Run from the `2026-fulcher-extractor` repository root, or install both `2026-fulcher-extractor` and `fulcheranalyzer` in the active environment first.

## What this writes

The notebook writes an ignored local run folder:

`local/runs/<RUN_ID>-qc-style-consistency-<SHOT>-fr<FRAME>/`

Main outputs:

- `qc_region_figures/<shot>_fr_<frame>_600_630.png`
- `qc_line_figures/<shot>_fr_<frame>_line_fits.pdf`
- `<shot>_fr_<frame>_boltzmann_qc.png`
- Boltzmann point/summary sidecar CSVs

Set `FULCHER_CUBE_DIR` if your SpectroCube folder is not under the default Dropbox location. Set `RUN_ID` to reproduce a named run folder.

In [ ]:
from __future__ import annotations

from datetime import datetime
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fulcher_analyzer import (
    BoltzmannPlot,
    apply_boltzmann_qc_mask,
    boltzmann_qc_points,
    read_intensities,
)
from fulcher_extractor.extract import extract_lines
from fulcher_extractor.fit import FitConfig
from fulcher_extractor.line_database import load_lines
from fulcher_extractor.line_policy import load_line_policy_set, overview_qc_lines
from fulcher_extractor.output import results_to_dataframe, write_fulcheranalyzer_csvs
from fulcher_extractor.qc import plot_region, write_line_fit_qc
from fulcher_extractor.spectrocube_io import load_spectrum

In [ ]:
SHOT = 193809
FRAME = 9
MAX_FIT_RELERR = 1.0

repo_root = Path.cwd()
default_cube_dir = (
    Path.home()
    / "Library/CloudStorage/Dropbox/Experiments/2025-LHD-BH/Echelle/20250926-spectrocubes"
)
cube_dir = Path(os.environ.get("FULCHER_CUBE_DIR", default_cube_dir))
cube_path = cube_dir / f"{SHOT}_Echelle_spectrocube_wmsr_403nm.nc"

run_id = os.environ.get("RUN_ID", datetime.now().strftime("%Y%m%d-%H%M"))
run_dir = repo_root / "local" / "runs" / f"{run_id}-qc-style-consistency-{SHOT}-fr{FRAME}"
run_dir

In [ ]:
if not cube_path.exists():
    raise FileNotFoundError(
        f"SpectroCube not found: {cube_path}\n"
        "Set FULCHER_CUBE_DIR to the folder containing the cube."
    )

spectrum = load_spectrum(cube_path, frame=FRAME)
lines = load_lines()
policy_set = load_line_policy_set()
label_lines = overview_qc_lines(lines, policy_set=policy_set)

config = FitConfig(
    line_left_width_nm=0.24,
    line_right_width_nm=0.18,
    center_left_offset_nm=0.16,
    center_right_offset_nm=0.08,
    instrument_sigma_nm=0.0273,
    instrument_sigma_leeway_nm=0.015,
    close_neighbor_threshold_nm=0.15,
)
results = extract_lines(spectrum, lines=lines, config=config)

intensity_dir = run_dir / "intensities"
fit_report_dir = run_dir / "fit_reports"
qc_region_dir = run_dir / "qc_region_figures"
qc_line_dir = run_dir / "qc_line_figures"
for path in (intensity_dir, fit_report_dir, qc_region_dir, qc_line_dir):
    path.mkdir(parents=True, exist_ok=True)

_, _, fit_report_path = write_fulcheranalyzer_csvs(
    results,
    output_dir=intensity_dir,
    shot=SHOT,
    frame=FRAME,
    metadata={"policy_layer": "line_policies.toml"},
)
fit_table = results_to_dataframe(results)
fit_report = fit_report_dir / fit_report_path.name
fit_table.to_csv(fit_report, index=False)

summary = (
    fit_table.groupby(["legacy_policy", "legacy_matrix_action"], dropna=False)
    .size()
    .rename("n_lines")
    .reset_index()
)
summary.to_csv(run_dir / "policy_summary.csv", index=False)

region_fig = plot_region(
    spectrum,
    lines=lines,
    label_lines=label_lines,
    guide_lines=label_lines,
    output_path=qc_region_dir / f"{SHOT}_fr_{FRAME}_600_630.png",
)
plt.close(region_fig)

line_paths = write_line_fit_qc(
    spectrum,
    results,
    pdf_path=qc_line_dir / f"{SHOT}_fr_{FRAME}_line_fits.pdf",
    columns=5,
)

fit_report, line_paths[0]

In [ ]:
intensity, error = read_intensities(SHOT, FRAME, data_folder=intensity_dir)
missing = (intensity <= 0.0) | (error <= 0.0)
intensity = intensity.mask(missing)
error = error.mask(missing)

bp = BoltzmannPlot((intensity, error), isotop="h")
points = boltzmann_qc_points(
    bp,
    max_fit_relerr=MAX_FIT_RELERR,
    fit_report=fit_report,
)
apply_boltzmann_qc_mask(bp, points, max_fit_relerr=MAX_FIT_RELERR)
bp.autofit()

fig = bp.plot_qc(
    points,
    title=f"H2 Fulcher-alpha double-exponent Boltzmann plot, shot {SHOT} frame {FRAME}",
    max_fit_relerr=MAX_FIT_RELERR,
    ylim=(1e-6, 2.0),
)
boltzmann_png = run_dir / f"{SHOT}_fr_{FRAME}_boltzmann_qc.png"
fig.savefig(boltzmann_png, dpi=220, bbox_inches="tight")
plt.close(fig)

points.to_csv(run_dir / f"{SHOT}_fr_{FRAME}_boltzmann_qc_points.csv", index=False)
pd.DataFrame(
    [
        {
            "shot": SHOT,
            "frame": FRAME,
            "alpha": bp.alpha,
            "alpha_err": bp.err[0],
            "beta": bp.beta,
            "beta_err": bp.err[1],
            "Trot1_K": bp.trot1,
            "Trot1_err_K": bp.err[2],
            "Trot2_K": bp.trot2,
            "Trot2_err_K": bp.err[3],
            "popt": ",".join(f"{value:.12g}" for value in bp.popt),
            "perr": ",".join(f"{value:.12g}" for value in bp.err),
        }
    ]
).to_csv(run_dir / f"{SHOT}_fr_{FRAME}_boltzmann_qc_summary.csv", index=False)
bp.trotall.to_csv(run_dir / f"{SHOT}_fr_{FRAME}_boltzmann_qc_trot.csv")
pd.DataFrame(
    {
        "band": [f"{index}-{index}" for index in range(len(bp.nd_vibrofit))],
        "nd_vibrofit": bp.nd_vibrofit,
    }
).to_csv(run_dir / f"{SHOT}_fr_{FRAME}_boltzmann_qc_nd_vibrofit.csv", index=False)

boltzmann_png

In [ ]:
q2 = points.loc[(points["band"] == "1-1") & (points["N"] == 2)]
display(q2[["line_id", "band", "N", "relerr", "fit_mask", "boltzmann_fit_action", "boltzmann_fit_reason"]])

print(f"run_dir: {run_dir}")
print(f"overview: {qc_region_dir / f'{SHOT}_fr_{FRAME}_600_630.png'}")
print(f"line QC: {qc_line_dir / f'{SHOT}_fr_{FRAME}_line_fits.pdf'}")
print(f"Boltzmann QC: {boltzmann_png}")
print(
    f"alpha={bp.alpha:.6g} beta={bp.beta:.6g} "
    f"Trot1={bp.trot1:.6g} K Trot2={bp.trot2:.6g} K"
)